### Tutorial: Logging, Validation, and Handling Failure States

In this tutorial, we'll add simple logging to track what our agents do and
handle failures gracefully. This is what makes agents ready for real-world use.

Learning Objectives:
- Add simple logging to track agent operations
- Handle agent failures gracefully
- Create basic monitoring for production use
- Build reliable and debuggable agents

In [11]:
# Install required packages
# pip install pydantic-ai openai python-dotenv

import os
import logging
from typing import List
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
from dotenv import load_dotenv

In [12]:
load_dotenv()

True

### Simple Logging Setup

Let's set up basic Python logging.

In [13]:
# Set up simple logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('agent.log'),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger('agent')

In [14]:
def log_action(action: str, success: bool = True):
    """Simple logging function."""
    if success:
        logger.info(f"✅ {action}")
        print(f"✅ {action}")
    else:
        logger.error(f"❌ {action}")
        print(f"❌ {action}")

### Agent with Logging

Let's create an agent that logs everything it does.

In [15]:
class LoggedResponse(BaseModel):
    answer: str = Field(description="The main answer")
    actions_logged: int = Field(description="Number of actions logged")

logged_agent = Agent('openai:gpt-4o', output_type=LoggedResponse)


In [16]:
@logged_agent.tool
async def logged_search(ctx: RunContext[None], query: str) -> str:
    """Search tool with logging."""
    log_action(f"Starting search for: {query}")
    
    try:
        import random
        if random.random() < 0.3:
            raise Exception("Search timeout")
        
        log_action(f"Search completed for: {query}")
        return f"Search results for: {query}"
    except Exception as e:
        log_action(f"Search failed for {query}: {e}", success=False)
        return f"Search unavailable for: {query}"

In [17]:
@logged_agent.tool
async def logged_calculation(ctx: RunContext[None], expression: str) -> str:
    """Calculator with logging."""
    log_action(f"Calculating: {expression}")
    
    try:
        result = eval(expression)
        log_action(f"Calculation successful: {expression} = {result}")
        return f"{expression} = {result}"
    except Exception as e:
        log_action(f"Calculation failed: {expression} - {e}", success=False)
        return f"Cannot calculate: {expression}"

### Failure Tracking

Let's track failures for monitoring.

In [18]:
failures = []

def track_failure(operation: str, error: str):
    """Track failures for monitoring."""
    failure = {
        'operation': operation,
        'error': error,
        'count': len(failures) + 1
    }
    failures.append(failure)
    log_action(f"Failure #{failure['count']}: {operation} - {error}", success=False)


In [19]:
@logged_agent.tool
async def monitored_operation(ctx: RunContext[None], task: str) -> str:
    """Operation with failure monitoring."""
    log_action(f"Starting {task}")
    
    try:
        import random
        if random.random() < 0.4:
            raise Exception(f"{task} service down")
        
        log_action(f"Completed {task}")
        return f"Successfully completed: {task}"
    except Exception as e:
        track_failure(task, str(e))
        return f"Failed to complete {task}, using fallback"


In [21]:
tests = [
    "Search for AI trends",
    "Calculate 20 + 30", 
    "Calculate 10 / 0",
    "Process data analysis",
    "Generate report"
]

for test in tests:
    print(f"\nTest: {test}")
    try:
        result = await logged_agent.run(test)
        print(f"Actions logged: {result.output.actions_logged}")
    except Exception as e:
        log_action(f"Agent failed: {e}", success=False)
    


Test: Search for AI trends


2025-09-24 14:00:41,020 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-09-24 14:00:41,023 - INFO - ✅ Starting search for: AI trends 2023
2025-09-24 14:00:41,024 - ERROR - ❌ Search failed for AI trends 2023: Search timeout


✅ Starting search for: AI trends 2023
❌ Search failed for AI trends 2023: Search timeout


2025-09-24 14:00:43,690 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Actions logged: 1

Test: Calculate 20 + 30


...

Actions logged: 1

Test: Generate report


2025-09-24 14:00:54,696 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-09-24 14:00:54,703 - INFO - ✅ Starting search for: How to generate an effective report
2025-09-24 14:00:54,704 - INFO - ✅ Search completed for: How to generate an effective report
2025-09-24 14:00:54,705 - INFO - ✅ Starting Verify document creation procedures
2025-09-24 14:00:54,708 - ERROR - ❌ Failure #2: Verify document creation procedures - Verify document creation procedures service down


✅ Starting search for: How to generate an effective report
✅ Search completed for: How to generate an effective report
✅ Starting Verify document creation procedures
❌ Failure #2: Verify document creation procedures - Verify document creation procedures service down


2025-09-24 14:00:57,674 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Actions logged: 2
